## 第 5 周 — 自我改进 RAG（mmaitsimwale）

### 练习目标
在第 5 周「专家知识工作者」基础上，把 **检索增强生成（RAG）** 做成可自评、可重试的闭环：生成 → LLM-as-judge → 按薄弱维度 refining。

### 第 5 周每天对应什么
- **第 1 天：** 关键字字典查找 RAG；模型 `gpt-4.1-nano`，尚无向量嵌入（Embeddings）。
- **第 2 天：** `RecursiveCharacterTextSplitter` → `HuggingFaceEmbeddings(all-MiniLM-L6-v2)` → Chroma `vector_db`；并用 t-SNE 可视化。
- **第 3 天：** LangChain `as_retriever()` + `ChatOpenAI`；基础系统提示（System Prompt）RAG。
- **第 4 天：** 评估工具 — 约 150 个测试用例、检索侧 MRR / nDCG / 关键字覆盖率、以及 LLM-as-judge（准确性 / 完整性 / 相关性）。
- **第 5 天：** 不用 LangChain 的高级 RAG：LLM 分块（`headline + summary + original_text`）→ `text-embedding-3-large` → Chroma `PersistentClient` → LLM 重排序（Rerank）→ 查询重写（Query Rewrite）。

### 本练习新增：自我改进循环（Self-improvement Loop）
对每个查询：
1. 重写查询 → 检索 → 重排序 → 生成答案。
2. 用 LLM-as-judge 打分（准确性 / 完整性 / 相关性）。
3. 若任一分数 < 4.5/5：把评委反馈注入下一轮提示并重试。
4. 重复直到达到 `MAX_ITERATIONS`，或 **所有分数 ≥ 4.5/5**。

### 怎么跑
从仓库根目录打开本笔记本；需 `.env` 中有 `OPENAI_API_KEY`（嵌入、分块、评测都会用到）。首次建库含 LLM 分块，较慢，可复用已有 Chroma collection。


## 算法流程（对照第 4/5 天）

1. **LLM 引导分块（第 5 天）：** 让 LLM 把每个 Markdown 切成语义连贯的 `Chunk(headline, summary, original_text)`。入库内容含标题+摘要，检索信号比只存原文更强。
2. **密集检索（Dense Retrieval）：** 用 `text-embedding-3-large` 嵌入查询；Chroma 余弦相似度返回前 k 个块。
3. **LLM 重排序（`RankOrder`，第 5 天）：** 第二次 LLM 调用按与问题的相关性重排块，把最有用上下文顶到前面。
4. **查询重写（第 5 天）：** 嵌入前把用户问题改写成更贴知识库的短查询。
5. **LLM-as-judge（`AnswerEval`，第 4 天）：** 对照参考答案，对准确性 / 完整性 / 相关性打 1–5 分。
6. **自我完善循环：** 把评委反馈原样带进下一代提示，并聚焦最薄弱维度再生答案。


In [ ]:
# ========== 设置：导入、仓库根目录解析、超参数常量 ==========

# os / sys：环境与路径；json：读 JSONL 测试集；math：本练习可预留数值工具
import os
import sys
import json
import math
# Path：跨平台路径对象，后面拼 week5/knowledge-base 等
from pathlib import Path

# load_dotenv：把 .env 密钥读进环境变量（Environment Variables）
from dotenv import load_dotenv
# Markdown / display：在 Jupyter 里漂亮展示文本（本笔记本可选用）
from IPython.display import Markdown, display
# OpenAI：官方客户端，本练习主要用于 embeddings.create
from openai import OpenAI
# Pydantic：结构化输出（Chunk / RankOrder / AnswerEval 等）
from pydantic import BaseModel, Field
# PersistentClient：Chroma 持久化向量库客户端
from chromadb import PersistentClient
# litellm.completion：统一聊天补全接口（可接多厂商，这里用 MODEL）
from litellm import completion
# tqdm：长循环进度条（LLM 分块 / 批量嵌入）
from tqdm import tqdm


def _repo_root() -> Path:
    """从当前工作目录向上找，直到发现 week5/knowledge-base（与 Week4_EXERCISE 相同约定）。"""
    # 先看 cwd，再看所有父目录
    for cand in [Path.cwd(), *Path.cwd().parents]:
        # 命中知识库目录即视为仓库根
        if (cand / "week5" / "knowledge-base").is_dir():
            return cand
    # 找不到就抛错；英文文案保持原样（依赖用户按提示启动）
    raise RuntimeError(
        "Cannot find week5/knowledge-base — start Jupyter from the llm_engineering repo root."
    )


# 解析出仓库根，后续所有相对资源都从这里拼
REPO_ROOT = _repo_root()
# Insurellm 知识库 Markdown 根目录
KNOWLEDGE_BASE = REPO_ROOT / "week5" / "knowledge-base"
# 第 4 天风格测试集（JSONL，每行一题）
TESTS_FILE = REPO_ROOT / "week5" / "evaluation" / "tests.jsonl"
# 本练习独立向量库路径，避免覆盖课程默认 vector_db
DB_NAME = str(REPO_ROOT / "week5" / "vector_db_exercise")   # separate from course vector_db

# 聊天 / 结构化输出所用模型 id（勿改，需与账号可用模型一致）
MODEL = "gpt-4.1-nano"
# 稠密嵌入模型 id
EMBEDDING_MODEL = "text-embedding-3-large"
# Chroma collection 名
COLLECTION_NAME = "docs_exercise"
# 向量检索先取 top-k，再交给 LLM 重排
RETRIEVAL_K = 10
# 每个问题最多自我改进轮数
MAX_ITERATIONS = 3          # max self-improvement passes per question
# 三维分数都要 ≥ 该阈值才算收敛（满分 5）
SUCCESS_THRESHOLD = 4.5     # minimum score (out of 5) on all dimensions

# 打印关键路径，方便确认 cwd / 启动位置对不对
print(f"Repo root  : {REPO_ROOT}")
print(f"KB path    : {KNOWLEDGE_BASE}")
print(f"Tests file : {TESTS_FILE}")
print(f"Vector DB  : {DB_NAME}")


In [ ]:
# ========== 从仓库根加载 .env（与 week5/day5.ipynb 相同的密钥检查模式）==========

# override=True：环境里已有同名变量时也以 .env 为准
load_dotenv(REPO_ROOT / ".env", override=True)

# 读 OpenAI / Anthropic / Google 密钥；后两者本格主要做存在性提示
openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")

# OpenAI：嵌入、分块、评测都依赖它
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set — required for embeddings, chunking and evaluation")

# Anthropic：可选
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (optional)")

# Google：可选
if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (optional)")


In [ ]:
# ========== OpenAI 客户端 ==========
# 用于 embeddings；聊天补全走 litellm.completion（读环境变量里的 OPENAI_API_KEY）
openai_client = OpenAI()


In [ ]:
# ========== Pydantic 模型：第 5 天分块/重排 + 第 4 天评估 ==========


class Result(BaseModel):
    """镜像 LangChain Document 的最小结构，方便下游统一用 page_content / metadata。"""
    # 块正文（检索与拼进 prompt 的文本）
    page_content: str
    # 来源路径、文档类型等元数据
    metadata: dict


class Chunk(BaseModel):
    """LLM 生成的语义块（day5：headline + summary + original_text）。"""
    # Field.description 会进 structured output schema，供模型理解字段含义——保持英文原样
    headline: str = Field(
        description="Brief heading for this chunk (a few words), optimised for retrieval"
    )
    summary: str = Field(
        description="A few sentences summarising this chunk to answer common questions"
    )
    original_text: str = Field(
        description="The original text of this chunk, exactly as-is, unchanged"
    )

    def as_result(self, document: dict) -> Result:
        # 拼成「标题 + 摘要 + 原文」，增强检索信号
        content = self.headline + "\n\n" + self.summary + "\n\n" + self.original_text
        return Result(
            page_content=content,
            # 继承文档级 source / type
            metadata={"source": document["source"], "type": document["type"]},
        )


class Chunks(BaseModel):
    # LLM 一次返回多个 Chunk 的容器
    chunks: list[Chunk]


class RankOrder(BaseModel):
    """按相关性排序后的 chunk ID 列表（day5 重排 structured output）。"""
    order: list[int] = Field(
        description="Chunk IDs from most relevant to least relevant"
    )


class AnswerEval(BaseModel):
    """LLM-as-judge 的分数与文字反馈（day4 评估模式）。"""
    feedback: str = Field(
        description="Concise feedback on answer quality vs reference answer"
    )
    accuracy: float = Field(
        description="Factual correctness: 1 (wrong) to 5 (perfect)"
    )
    completeness: float = Field(
        description="Coverage of all required information: 1 (very poor) to 5 (ideal)"
    )
    relevance: float = Field(
        description="How directly the answer addresses the question: 1 (off-topic) to 5 (ideal)"
    )


class TestQuestion(BaseModel):
    """带参考答案的测试题（day4 JSONL 一行对应一题）。"""
    question: str
    keywords: list[str]
    reference_answer: str
    category: str


In [ ]:
# ========== 摄取管道：读文档 → LLM 分块 → 嵌入 → 写入 Chroma ==========

# 粗略估计目标块大小，用来提示 LLM「大约切成几段」
AVERAGE_CHUNK_SIZE = 500   # hint for how many chunks the LLM should produce


def fetch_documents() -> list[dict]:
    """从知识库加载全部 Markdown（day5：按子文件夹名当 doc type）。"""
    documents = []
    # 遍历 knowledge-base 下一级目录（employees / products / ...）
    for folder in KNOWLEDGE_BASE.iterdir():
        if not folder.is_dir():
            continue
        # 文件夹名即文档类型标签
        doc_type = folder.name
        # 递归收集该类型下所有 .md
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append(
                    {"type": doc_type, "source": file.as_posix(), "text": f.read()}
                )
    print(f"Loaded {len(documents)} documents from {KNOWLEDGE_BASE}")
    return documents


def make_chunking_prompt(document: dict) -> str:
    """构造 LLM 分块提示（day5：含重叠 overlap 提示）。prompt 正文保持英文原样。"""
    # 按平均块长估算期望块数（+1 避免低估）
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and split it into overlapping chunks for a Knowledge Base.

The document is from the shared drive of a company called Insurellm.
Document type : {document["type"]}
Document path : {document["source"]}

A chatbot will use these chunks to answer questions about the company.
Divide the document as you see fit; the entire document must be covered.
This document should probably be split into approximately {how_many} chunks (more or less is fine).
Include ~25% overlap (~50 words) between adjacent chunks for better retrieval.

For each chunk provide:
  - headline : a brief heading most likely to surface in a query
  - summary  : a few sentences summarising the chunk
  - original_text : the original text of the chunk, exactly as-is

Together, your chunks must represent the entire document.

Document text:
{document["text"]}

Respond with the chunks.
"""


def process_document(document: dict) -> list[Result]:
    """对单文档做 LLM 引导分块，并转成 Result 列表。"""
    # 用户消息里塞完整分块 prompt
    messages = [{"role": "user", "content": make_chunking_prompt(document)}]
    # response_format=Chunks：要求模型按 Pydantic schema 返回
    response = completion(model=MODEL, messages=messages, response_format=Chunks)
    # 解析 JSON → Chunks → 各 Chunk
    chunks = Chunks.model_validate_json(response.choices[0].message.content).chunks
    # 每个 Chunk 拼成带 metadata 的 Result
    return [chunk.as_result(document) for chunk in chunks]


def create_chunks(documents: list[dict]) -> list[Result]:
    """对全部文档调用 LLM 分块；单文档失败只警告，不中断整批。"""
    all_chunks: list[Result] = []
    for doc in tqdm(documents, desc="LLM chunking"):
        try:
            all_chunks.extend(process_document(doc))
        except Exception as e:
            # 打印失败来源，继续下一篇
            print(f"  Warning: could not chunk {doc['source']}: {e}")
    return all_chunks


def build_vectorstore(force_rebuild: bool = False) -> PersistentClient:
    """构建或复用 Chroma 向量库。

    若 collection 已有向量则跳过重建（LLM 分块+嵌入很慢）。
    传入 force_rebuild=True 可强制清空重做。
    """
    # 打开（或创建）持久化客户端，path=DB_NAME
    chroma = PersistentClient(path=DB_NAME)
    existing = [c.name for c in chroma.list_collections()]

    # 已存在且非强制重建：若有数据则直接复用
    if COLLECTION_NAME in existing and not force_rebuild:
        col = chroma.get_collection(COLLECTION_NAME)
        if col.count() > 0:
            print(f"Loaded existing collection '{COLLECTION_NAME}' ({col.count()} vectors). "
                  f"Pass force_rebuild=True to rebuild.")
            return chroma

    # 强制重建时先删旧 collection
    if COLLECTION_NAME in existing:
        chroma.delete_collection(COLLECTION_NAME)

    # 读盘 → LLM 分块
    documents = fetch_documents()
    chunks = create_chunks(documents)

    # 准备写入 Chroma 的文本与元数据
    texts = [c.page_content for c in chunks]
    metas = [c.metadata for c in chunks]

    # 批量嵌入以遵守 API 限速/请求体大小限制
    batch_size = 100
    all_vectors: list = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch = texts[i : i + batch_size]
        emb_data = openai_client.embeddings.create(
            model=EMBEDDING_MODEL, input=batch
        ).data
        # 按 batch 顺序追加向量
        all_vectors.extend([e.embedding for e in emb_data])

    # 创建 collection 并一次性 add
    col = chroma.get_or_create_collection(COLLECTION_NAME)
    ids = [str(i) for i in range(len(chunks))]
    col.add(ids=ids, embeddings=all_vectors, documents=texts, metadatas=metas)
    print(f"Built '{COLLECTION_NAME}' with {col.count()} vectors.")
    return chroma


# 构建（或加载）向量库，并取出本练习用的 collection
chroma_client = build_vectorstore()
collection = chroma_client.get_collection(COLLECTION_NAME)


In [ ]:
# ========== 检索：向量搜索 → LLM 重排序（day5）==========


def fetch_context_unranked(question: str, k: int = RETRIEVAL_K) -> list[Result]:
    """把问题嵌入后，按余弦相似度取 top-k 块（尚未重排）。"""
    # 单条查询向量
    query_vec = (
        openai_client.embeddings.create(model=EMBEDDING_MODEL, input=[question])
        .data[0]
        .embedding
    )
    # Chroma 近邻查询
    results = collection.query(query_embeddings=[query_vec], n_results=k)
    # 打包成 Result，方便后续统一处理
    return [
        Result(page_content=doc, metadata=meta)
        for doc, meta in zip(results["documents"][0], results["metadatas"][0])
    ]


def rerank(question: str, chunks: list[Result]) -> list[Result]:
    """用 RankOrder 结构化输出做 LLM 重排（把最相关块排到前面）。"""
    # system / user prompt 字符串影响行为，保持英文原样
    system_prompt = (
        "You are a document re-ranker.\n"
        "Rank the provided chunks by relevance to the question, most relevant first.\n"
        "Reply only with the ordered list of chunk IDs (integers), nothing else."
    )
    user_prompt = f"Question: {question}\n\nChunks:\n\n"
    # 给模型看的 ID 从 1 开始编号
    for idx, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {idx + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the ranked chunk IDs, nothing else."

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    # 强制按 RankOrder schema 返回
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    order = RankOrder.model_validate_json(response.choices[0].message.content).order
    # 防止模型吐出越界索引；ID 是 1-based，取列表时要 -1
    return [chunks[i - 1] for i in order if 1 <= i <= len(chunks)]


def fetch_context(question: str) -> list[Result]:
    """完整检索管线：先向量 top-k，再 LLM 重排。"""
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)


In [ ]:
# ========== 查询重写：把用户问题收成更利于检索的短查询（day5）==========


def rewrite_query(question: str, prior_answers: list[str] | None = None) -> str:
    """把用户问题改写成适合知识库检索的短 query。

    prior_answers：同会话里前几轮答案，供重写时消歧（可为空）。
    """
    # None → 空列表，避免后面 join 报错
    prior_answers = prior_answers or []
    # 只带最近 2 条答案当上下文，控制 prompt 长度
    history_text = "\n".join(prior_answers[-2:]) if prior_answers else "(none)"
    # 整段说明放进 system；要求模型只回一条搜索串——prompt 保持英文
    message = f"""
You are answering questions about the company Insurellm.
You are about to search a Knowledge Base.

Prior answers in this conversation (for context):
{history_text}

User's current question:
{question}

Respond ONLY with a single, short, specific query string that will surface the most
relevant content in the Knowledge Base.
- Focus on the question details.
- Do not mention the company name unless it is a general company question.

IMPORTANT: Reply with ONLY the search query, nothing else.
"""
    response = completion(
        model=MODEL, messages=[{"role": "system", "content": message}]
    )
    # strip 去掉首尾空白/换行
    return response.choices[0].message.content.strip()


In [ ]:
# ========== 评估助手（内联实现，不依赖 week5/evaluation 模块导入路径）==========


def load_tests() -> list[TestQuestion]:
    """从 JSONL 加载测试题：每行一个 JSON 对象。"""
    tests: list[TestQuestion] = []
    with open(TESTS_FILE, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            # 跳过空行
            if line:
                # ** 解包字段进 TestQuestion
                tests.append(TestQuestion(**json.loads(line)))
    print(f"Loaded {len(tests)} test questions from {TESTS_FILE}")
    return tests


def evaluate_answer_llm(
    question: str, generated: str, reference: str
) -> AnswerEval:
    """LLM-as-judge：对照参考答案打分（对齐 eval.py 约 130–158 行思路）。"""
    # judge_messages 里的英文评分标准必须原样保留
    judge_messages = [
        {
            "role": "system",
            "content": (
                "You are an expert evaluator assessing the quality of answers. "
                "Evaluate the generated answer by comparing it to the reference answer. "
                "Only give 5/5 scores for perfect answers."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Question:\n{question}\n\n"
                f"Generated Answer:\n{generated}\n\n"
                f"Reference Answer:\n{reference}\n\n"
                "Please evaluate the generated answer on three dimensions:\n"
                "1. Accuracy: How factually correct is it compared to the reference? "
                "Only give 5/5 for perfect answers. If the answer is wrong, accuracy must be 1.\n"
                "2. Completeness: How thoroughly does it address all aspects, covering all "
                "information from the reference answer?\n"
                "3. Relevance: How well does it directly answer the specific question, "
                "giving no additional information?\n\n"
                "Provide detailed feedback and scores from 1 (very poor) to 5 (ideal) "
                "for each dimension."
            ),
        },
    ]
    # 强制按 AnswerEval schema 返回 JSON
    resp = completion(model=MODEL, messages=judge_messages, response_format=AnswerEval)
    return AnswerEval.model_validate_json(resp.choices[0].message.content)


def identify_weakness(eval_result: AnswerEval) -> str:
    """返回得分最低的维度名（accuracy / completeness / relevance），供下一轮聚焦改进。"""
    scores = {
        "accuracy": eval_result.accuracy,
        "completeness": eval_result.completeness,
        "relevance": eval_result.relevance,
    }
    # min + key：按分值取最弱键名
    return min(scores, key=lambda k: scores[k])


In [ ]:
# ========== 答案生成：带迭代感知的 System Prompt（第 3/5 天模式）==========

# 首轮：只用上下文答题；要求不幻觉——prompt 正文保持英文
SYSTEM_PROMPT_BASE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
Your answer will be evaluated for accuracy, relevance, and completeness.
Answer ONLY from the provided context. If the answer is not in the context, say so.
Do NOT hallucinate. Be precise and complete.

Context (extracts from the Knowledge Base):
{context}

With this context, answer the user's question accurately, completely, and relevantly.
"""

# 后续轮：注入上一轮分数、反馈与薄弱维度，引导针对性改进
SYSTEM_PROMPT_REFINE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
Your answer will be evaluated for accuracy, relevance, and completeness.
Answer ONLY from the provided context. Do NOT hallucinate. Be precise and complete.

A previous attempt at this question received these scores:
  Accuracy     : {accuracy}/5
  Completeness : {completeness}/5
  Relevance    : {relevance}/5

Evaluator feedback: {feedback}

Your primary improvement focus for this attempt: {weakness}.
Improve specifically on {weakness}. Do NOT include information not supported by the context.

Context (extracts from the Knowledge Base):
{context}

With this context, provide an improved answer.
"""


def make_rag_messages(
    question: str,
    chunks: list[Result],
    iteration: int = 1,
    prior_eval: AnswerEval | None = None,
    weakness: str = "",
) -> list[dict]:
    """组装发给 LLM 的 messages：system（含上下文/反馈）+ user（原问题）。"""
    # 每块标注来源路径，方便模型归因
    context = "\n\n".join(
        f"Extract from {c.metadata['source']}:\n{c.page_content}" for c in chunks
    )
    # 第 1 轮或没有先验评测 → 用 BASE；否则用 REFINE
    if iteration == 1 or prior_eval is None:
        system_content = SYSTEM_PROMPT_BASE.format(context=context)
    else:
        system_content = SYSTEM_PROMPT_REFINE.format(
            context=context,
            accuracy=prior_eval.accuracy,
            completeness=prior_eval.completeness,
            relevance=prior_eval.relevance,
            feedback=prior_eval.feedback,
            weakness=weakness,
        )
    return [
        {"role": "system", "content": system_content},
        {"role": "user", "content": question},
    ]


def generate_answer(
    question: str,
    chunks: list[Result],
    iteration: int = 1,
    prior_eval: AnswerEval | None = None,
    weakness: str = "",
) -> str:
    """调用 LLM，返回答案字符串。"""
    messages = make_rag_messages(question, chunks, iteration, prior_eval, weakness)
    resp = completion(model=MODEL, messages=messages)
    return resp.choices[0].message.content


In [ ]:
# ========== 自我完善循环：本练习的核心贡献 ==========


def _print_iteration(iteration: int, answer: str, ev: AnswerEval) -> None:
    # 打印单轮答案与三维分数，便于观察是否收敛
    print(f"\n{'='*64}")
    print(f"  Iteration {iteration}")
    print(f"{'='*64}")
    print(f"Answer:\n{answer}\n")
    print(
        f"Scores — Accuracy: {ev.accuracy:.1f}/5  "
        f"Completeness: {ev.completeness:.1f}/5  "
        f"Relevance: {ev.relevance:.1f}/5"
    )
    print(f"Feedback: {ev.feedback}")


def self_improving_answer(
    question: str,
    reference_answer: str,
    verbose: bool = True,
) -> dict:
    """
    迭代生成并改进 RAG 答案。

    每轮流程：
      1. 结合先前答案做查询重写。
      2. 检索 + 重排上下文块。
      3. 生成答案（第 1 轮：基础提示；第 2+ 轮：注入反馈）。
      4. 用 LLM-as-judge 对照参考答案评分。
      5. 若全部 ≥ SUCCESS_THRESHOLD 或达到 MAX_ITERATIONS 则停止。
      6. 否则记录反馈、找出最弱维度，进入下一轮。

    返回 dict：question / answer / eval / iteration / converged。
    """
    # 跨轮状态：先前答案、上一轮评测、本轮要主攻的薄弱维度
    prior_answers: list[str] = []
    prior_eval: AnswerEval | None = None
    weakness: str = ""
    final_result: dict = {}

    for iteration in range(1, MAX_ITERATIONS + 1):
        # 第 1 步：查询重写（可带上先前答案作消歧）
        query = rewrite_query(question, prior_answers)

        # 第 2 步：检索 + 重排序
        chunks = fetch_context(query)

        # 第 3 步：生成（第 1 轮朴素提示，之后由反馈引导）
        answer = generate_answer(question, chunks, iteration, prior_eval, weakness)

        # 第 4 步：对照参考答案做 LLM-as-judge
        eval_result = evaluate_answer_llm(question, answer, reference_answer)

        if verbose:
            _print_iteration(iteration, answer, eval_result)

        # 先写好本轮结果；后面可能把 converged 翻成 True
        final_result = {
            "question": question,
            "answer": answer,
            "eval": eval_result,
            "iteration": iteration,
            "converged": False,
        }

        # 第 5 步：成功条件——三维都过阈值
        scores = [eval_result.accuracy, eval_result.completeness, eval_result.relevance]
        if all(s >= SUCCESS_THRESHOLD for s in scores):
            final_result["converged"] = True
            if verbose:
                print(
                    f"\n  Converged at iteration {iteration} "
                    f"(all scores >= {SUCCESS_THRESHOLD}/5)"
                )
            break

        # 第 6 步：为下一轮准备状态
        prior_answers.append(answer)
        prior_eval = eval_result
        weakness = identify_weakness(eval_result)
        if verbose and iteration < MAX_ITERATIONS:
            print(f"\n  -> Retrying: focusing improvement on '{weakness}'")

    return final_result


In [ ]:
# ========== 示例运行：对一小批测试题跑自我改进 RAG ==========
# 增大 SAMPLE_SIZE 可评估更多题；全套约 150 题。

# 加载 JSONL 测试集
tests = load_tests()
# 演示用样本量（控制 API 成本与耗时）
SAMPLE_SIZE = 5
SAMPLE = tests[:SAMPLE_SIZE]

print(f"Running self-improving RAG on {SAMPLE_SIZE} test questions "
      f"(up to {MAX_ITERATIONS} iterations each, threshold {SUCCESS_THRESHOLD}/5).\n")

# 收集每题最终结果，供下一格汇总表使用
all_results: list[dict] = []
for idx, t in enumerate(SAMPLE):
    print(f"\n{'#' * 64}")
    print(f"  Q{idx + 1}: {t.question}  [{t.category}]")
    print(f"{'#' * 64}")
    # 核心：自我改进循环
    result = self_improving_answer(t.question, t.reference_answer)
    # 附带类别，方便后面按 category 展示
    result["category"] = t.category
    all_results.append(result)


In [ ]:
# ========== 结果汇总表：每题轮次、三维分数、是否收敛 ==========

print(f"\n{'=' * 70}")
print(f"{'RESULTS SUMMARY':^70}")
print(f"{'=' * 70}")
# 表头：题号、类别、迭代次数、Acc/Comp/Rel、是否收敛
header = (
    f"  {'Q':<2} {'Category':<18} {'Iter':>4}  "
    f"{'Acc':>5} {'Comp':>5} {'Rel':>5}  {'Converged'}"
)
print(header)
print("-" * 70)

# 逐题打印一行
for i, r in enumerate(all_results, 1):
    ev = r["eval"]
    converged_str = "YES" if r["converged"] else "NO"
    print(
        f"  {i:<2} {r['category']:<18} {r['iteration']:>4}  "
        f"{ev.accuracy:>5.1f} {ev.completeness:>5.1f} {ev.relevance:>5.1f}  "
        f"{converged_str}"
    )

print("-" * 70)

# 样本均值（Accuracy / Completeness / Relevance）
avg_acc  = sum(r["eval"].accuracy      for r in all_results) / len(all_results)
avg_comp = sum(r["eval"].completeness  for r in all_results) / len(all_results)
avg_rel  = sum(r["eval"].relevance     for r in all_results) / len(all_results)
# 收敛题数
n_conv   = sum(1 for r in all_results if r["converged"])

print(
    f"  {'AVERAGES':<22}      {avg_acc:>5.1f} {avg_comp:>5.1f} {avg_rel:>5.1f}"
)
print(f"\n  Converged: {n_conv}/{len(all_results)} questions reached "
      f">= {SUCCESS_THRESHOLD}/5 on all dimensions.")
# 回顾本练习叠了哪些第 5 天技巧
print(f"\n  Key improvements applied:")
print(f"    - LLM-guided chunking (headline + summary + original text)")
print(f"    - text-embedding-3-large for dense retrieval")
print(f"    - LLM reranking (RankOrder structured output)")
print(f"    - Query rewriting before each retrieval")
print(f"    - Iterative self-improvement driven by evaluator feedback")
